# Fuzzy Logic Application: Air Conditioner Fan-Speed Controller

## 1. Introduction

Fuzzy Logic is a method of reasoning that handles values between completely true and completely false. Unlike classical Boolean logic, which uses only 0 and 1, fuzzy logic allows a degree of membership between 0 and 1.

This notebook applies fuzzy logic to a simple air-conditioner fan-speed controller. Temperature is represented using fuzzy sets named **Low**, **Medium**, and **High**. A set of linguistic rules maps these temperature categories to fan-speed levels.

The original notebook used triangular membership functions and a weighted-average calculation. This improved version preserves the same application and output levels while correcting the high-temperature boundary behaviour and making the membership functions explicit.

## 2. Aim

To design and implement a fuzzy logic-based air-conditioner controller that determines fan speed according to the input temperature.

## 3. Learning Objectives

After completing this notebook, the learner should be able to:

- Explain fuzzy logic and fuzzy set membership.
- Distinguish crisp values from fuzzy membership degrees.
- Define and plot temperature membership functions.
- Apply a simple fuzzy rule base.
- Calculate a crisp fan-speed output using weighted-average defuzzification.
- Test the controller at normal and boundary temperatures.

## 4. Problem Statement

Design a controller that accepts a temperature in degrees Celsius and returns a fan-speed percentage. The controller should respond gradually to temperature changes rather than switching abruptly between fixed speed levels.

**Controller rule idea:**
- Low temperature → Low fan speed (20%).
- Medium temperature → Medium fan speed (50%).
- High temperature → High fan speed (100%).

**Scope:** This is an educational fuzzy-control model. It does not connect to physical air-conditioner hardware or account for humidity, occupancy, energy consumption, or compressor safety constraints.


## 5. Theory of Fuzzy Logic

### 5.1 Crisp logic and fuzzy logic

In classical logic, a statement is either true (1) or false (0). In fuzzy logic, a statement can have a truth or membership degree anywhere between 0 and 1.

For example, a temperature of 28°C may belong partly to the **Medium** set and partly to the **High** set. This overlap allows a controller to change its output smoothly.

### 5.2 Fuzzy sets and membership functions

A fuzzy set describes how strongly an input belongs to a linguistic category. A membership function maps an input value to a membership degree:

\[
\mu_A(x) \in [0,1]
\]

Here, \(x\) is the input temperature and \(A\) is a fuzzy set such as Low, Medium, or High.

### 5.3 Triangular membership function

A triangular membership function is defined by three parameters \(a\), \(b\), and \(c\), where \(a \leq b \leq c\). It rises from zero to one and then falls to zero. It is useful for representing a category with a central representative value.

### 5.4 Shoulder membership functions

A left-shoulder function remains fully active at the lower end of the input range and then decreases. A right-shoulder function increases toward the upper end and remains fully active beyond it. Shoulder functions are useful for boundary categories such as very low and very high temperature.

### 5.5 Fuzzification

Fuzzification converts a crisp temperature into membership degrees for the fuzzy sets. A single temperature can have non-zero membership in more than one set.

### 5.6 Rule evaluation and output calculation

The controller uses three rules:

1. IF temperature is Low, THEN fan speed is 20%.
2. IF temperature is Medium, THEN fan speed is 50%.
3. IF temperature is High, THEN fan speed is 100%.

This notebook uses a **weighted-average (zero-order Sugeno-style) output calculation**. Each rule has a fixed output value, and its membership degree acts as the weight:

\[
\text{Fan Speed} =
\frac{\sum_i \mu_i s_i}{\sum_i \mu_i}
\]

where \(\mu_i\) is the membership degree of rule \(i\), and \(s_i\) is its assigned fan-speed value.

This is a weighted-average singleton method, not a full Mamdani max-min aggregation and centroid calculation.


## 6. Importing Required Libraries

NumPy is used for numerical operations and Matplotlib is used to visualise membership functions and controller behaviour. These are the only external libraries required.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

print("Libraries imported successfully.")

## 7. Defining the Temperature Membership Functions

The input universe is 0°C to 40°C, matching the range used in the original notebook.

The original high-temperature triangle was defined as `(25, 40, 40)`. At exactly 40°C, the triangular function returned zero because its right-hand side had zero width. This caused all three rule memberships to become zero at the upper boundary, and the controller returned 0% fan speed.

To correct this boundary issue, this implementation uses:
- **Low:** a left-shoulder function, fully active from 0°C to 10°C and decreasing to zero at 25°C.
- **Medium:** a triangular function with parameters (15, 25, 35).
- **High:** a right-shoulder function, increasing from 25°C to 40°C and fully active at and above 40°C.

The low and high functions are deliberately shoulder-shaped so that the controller has meaningful membership at the ends of the input range.


In [ ]:
def triangular(x, a, b, c):
    """Return triangular membership degrees for a <= b <= c, with a < c."""
    if not (a <= b <= c) or a == c:
        raise ValueError("Parameters must satisfy a <= b <= c and a < c.")

    x = np.asarray(x, dtype=float)
    rising = (x - a) / (b - a) if b != a else np.ones_like(x)
    falling = (c - x) / (c - b) if c != b else np.ones_like(x)
    membership = np.maximum(np.minimum(rising, falling), 0.0)
    return np.clip(membership, 0.0, 1.0)


def left_shoulder(x, plateau_end, zero_at):
    """Membership is 1 up to plateau_end, then decreases to 0 at zero_at."""
    if plateau_end >= zero_at:
        raise ValueError("plateau_end must be smaller than zero_at.")

    x = np.asarray(x, dtype=float)
    membership = np.where(
        x <= plateau_end,
        1.0,
        np.where(x >= zero_at, 0.0, (zero_at - x) / (zero_at - plateau_end))
    )
    return np.clip(membership, 0.0, 1.0)


def right_shoulder(x, zero_at, plateau_start):
    """Membership is 0 up to zero_at, then increases to 1 at plateau_start."""
    if zero_at >= plateau_start:
        raise ValueError("zero_at must be smaller than plateau_start.")

    x = np.asarray(x, dtype=float)
    membership = np.where(
        x <= zero_at,
        0.0,
        np.where(x >= plateau_start, 1.0, (x - zero_at) / (plateau_start - zero_at))
    )
    return np.clip(membership, 0.0, 1.0)


temperature = np.arange(0, 41, 1)

low = left_shoulder(temperature, plateau_end=10, zero_at=25)
medium = triangular(temperature, 15, 25, 35)
high = right_shoulder(temperature, zero_at=25, plateau_start=40)

print("Temperature membership functions defined.")

## 8. Visualising Temperature Membership Functions

The graph shows how each temperature belongs to the Low, Medium, and High fuzzy sets. The overlap between the sets allows multiple rules to contribute to the final fan-speed calculation.


In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(temperature, low, label="Low Temperature")
plt.plot(temperature, medium, label="Medium Temperature")
plt.plot(temperature, high, label="High Temperature")

plt.xlabel("Temperature (°C)")
plt.ylabel("Membership Degree")
plt.title("Temperature Fuzzy Membership Functions")
plt.xlim(0, 40)
plt.ylim(0, 1.05)
plt.legend()
plt.grid(True)
plt.show()

## 9. Fuzzification

The fuzzification function accepts a crisp temperature and calculates its membership degree in each fuzzy set. The input is restricted to the defined universe of 0°C to 40°C so that out-of-range values are not silently treated as valid sensor readings.


In [ ]:
def fuzzify_temperature(temp):
    """Return Low, Medium, and High membership degrees for a temperature."""
    if not isinstance(temp, (int, float, np.integer, np.floating)):
        raise TypeError("Temperature must be a numeric value.")
    if not 0 <= temp <= 40:
        raise ValueError("Temperature must be between 0°C and 40°C.")

    return {
        "Low": float(left_shoulder(temp, plateau_end=10, zero_at=25)),
        "Medium": float(triangular(temp, 15, 25, 35)),
        "High": float(right_shoulder(temp, zero_at=25, plateau_start=40))
    }


example_temperature = 30
print(f"Fuzzification for {example_temperature}°C:")
for category, degree in fuzzify_temperature(example_temperature).items():
    print(f"{category}: {degree:.2f}")

## 10. Defining the Fuzzy Rule Base and Controller

Each fuzzy rule contributes a fixed fan-speed value. The rule's membership degree is used as its weight in the weighted-average calculation.

If no rule is active, the function raises an error rather than returning a misleading fan speed of zero. Within the defined 0°C–40°C input range, the shoulder functions ensure that at least one of the Low or High sets is active.


In [ ]:
FAN_SPEED_OUTPUTS = {
    "Low": 20.0,
    "Medium": 50.0,
    "High": 100.0
}


def fuzzy_controller(temp):
    """Calculate fan speed (%) using weighted-average fuzzy rule outputs."""
    degrees = fuzzify_temperature(temp)

    numerator = sum(
        degrees[category] * output
        for category, output in FAN_SPEED_OUTPUTS.items()
    )
    denominator = sum(degrees.values())

    if denominator == 0:
        raise RuntimeError("No fuzzy rule was activated for this temperature.")

    fan_speed = numerator / denominator
    return float(np.clip(fan_speed, 0.0, 100.0))


print("Fuzzy controller defined successfully.")

## 11. Testing the Controller at Selected Temperatures

The following test points include low, medium, high, and boundary temperatures. The expected behaviour is that the controller should return a valid fan-speed percentage and should not return zero at 40°C.


In [ ]:
test_temperatures = [0, 10, 15, 20, 25, 30, 35, 40]

print("Fuzzy Logic Air Conditioner Controller")
print("-" * 55)

for temp in test_temperatures:
    memberships = fuzzify_temperature(temp)
    speed = fuzzy_controller(temp)
    print(
        f"Temperature: {temp:>2}°C | "
        f"Low: {memberships['Low']:.2f} | "
        f"Medium: {memberships['Medium']:.2f} | "
        f"High: {memberships['High']:.2f} | "
        f"Fan Speed: {speed:.2f}%"
    )

## 12. Plotting the Controller Response

This graph displays the calculated fan speed across the complete input temperature range. The response is expected to increase as temperature moves from the Low region toward the High region.


In [ ]:
temperatures = np.arange(0, 41, 1)
fan_speeds = [fuzzy_controller(t) for t in temperatures]

plt.figure(figsize=(9, 5))
plt.plot(temperatures, fan_speeds, label="Calculated Fan Speed")
plt.scatter(test_temperatures, [fuzzy_controller(t) for t in test_temperatures],
            label="Test Temperatures")
plt.xlabel("Temperature (°C)")
plt.ylabel("Fan Speed (%)")
plt.title("Fuzzy Logic Temperature Control")
plt.xlim(0, 40)
plt.ylim(0, 105)
plt.legend()
plt.grid(True)
plt.show()

## 13. Automated Validation and Boundary Tests

The assertions below verify the main properties of the controller:

- Membership degrees remain between 0 and 1.
- The low-temperature boundary activates the Low set.
- The high-temperature boundary activates the High set.
- Fan speed remains between 0% and 100%.
- Fan speed at 40°C is 100%, correcting the original boundary defect.
- Invalid temperatures are rejected.


In [ ]:
# Membership values must always remain within [0, 1].
for temp in range(41):
    memberships = fuzzify_temperature(temp)
    assert all(0.0 <= degree <= 1.0 for degree in memberships.values())

# Boundary membership checks.
assert fuzzify_temperature(0)["Low"] == 1.0
assert fuzzify_temperature(40)["High"] == 1.0

# Controller output must remain in the permitted percentage range.
for temp in range(41):
    speed = fuzzy_controller(temp)
    assert 0.0 <= speed <= 100.0

# The upper boundary must activate high fan speed, not return 0%.
assert fuzzy_controller(40) == 100.0
assert fuzzy_controller(0) == 20.0

# Invalid inputs must be rejected.
for invalid_temp in (-1, 41):
    try:
        fuzzy_controller(invalid_temp)
    except ValueError:
        pass
    else:
        raise AssertionError(f"Invalid temperature {invalid_temp} was not rejected.")

print("All membership, boundary, output-range, and invalid-input tests passed.")

## 14. Observations

1. Fuzzy membership allows a temperature to belong partially to more than one category.
2. The Low, Medium, and High sets overlap, so adjacent rules can contribute to the same output.
3. The weighted-average method combines the rule outputs according to their membership degrees.
4. The shoulder functions provide valid membership at the lower and upper boundaries of the temperature range.
5. At 0°C, the Low set is fully active and the controller returns 20%.
6. At 40°C, the High set is fully active and the controller returns 100%.
7. The controller response changes gradually across the input range rather than using abrupt category switching.

## 15. Limitations

- The model uses only temperature as its input.
- The fan-speed outputs (20%, 50%, and 100%) are illustrative values.
- The rule base is intentionally small and does not represent a commercially engineered HVAC controller.
- The weighted-average singleton method is a simplified inference approach; it is not a full Mamdani fuzzy inference system.
- The model has not been tested against real sensor readings or physical hardware.

## 16. Conclusion

This notebook demonstrates a fuzzy logic-based air-conditioner fan-speed controller. Temperature is fuzzified into Low, Medium, and High categories, and a simple rule base maps these categories to fan-speed outputs. A weighted-average calculation produces the final crisp fan-speed percentage.

The original implementation's high-temperature boundary issue was corrected by using a right-shoulder membership function. Automated tests confirm that the controller produces a valid output throughout the defined temperature range and returns 100% fan speed at 40°C.

The application illustrates how fuzzy logic can model gradual decisions in situations where category boundaries are not strictly binary.

## 17. References

1. Zadeh, L. A. (1965). *Fuzzy sets*. Information and Control, 8(3), 338–353.
2. Ross, T. J. (2010). *Fuzzy Logic with Engineering Applications* (3rd ed.). Wiley.
3. NumPy documentation: https://numpy.org/doc/
4. Matplotlib documentation: https://matplotlib.org/stable/
